# SPY vs BTC — barrier control experiment

**Question:** does the price-only barrier model that failed on BTC do any better on SPY?
Change only the asset. Same features, model, folds, labels and verdict rules.

**Pre-registered design (fixed 21 Sep 2026, before any SPY result was seen)**
- Price-only features (OHLCV → the same 50-feature set), same transformer, `ENC_LEN=60`, 5 expanding walk-forward folds.
- Barriers scaled to each asset's own volatility: ±k × ATR(60), with **k = 4.5, 7.5, 15**
  (= BTC's old 0.3 / 0.5 / 1.0 % barriers in volatility terms). Hold **120 min**. Two seeds.
- Nothing dropped: timeouts exit at market with real P&L. SPY trades never cross the session close.
- **Primary metric: edge vs best constant (pp)** — cost-free, directly comparable across assets.
  EV after costs is secondary (costs differ hugely: BTC 0.085 % vs SPY 0.01 % round trip).
- **Pass bar:** positive edge in all 5 folds *and* positive EV, on both seeds.
- Reproduction check first (BTC only): the old ±0.3 % / 120 min row should land near the Trading (IV)
  result (edge ≈ −0.9 pp). This version also fixes the fold-alignment bug, so small differences are expected.

**How to run:** set `ASSET` in the config cell → *Run All*. For the other asset: change `ASSET`,
**Restart** the kernel, *Run All* again. Each run writes `results_<asset>.csv`; the last cell compares both.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

ASSET = "btc"          # <-- "btc" or "spy"

# --- data ---
BQ_PROJECT = "trading-brains"
BQ_TABLE   = "trading-brains.market_microstructure.mmt_btc_1m_v3"
SPY_CSV    = r"C:\Users\matth\Desktop\Trading\spy_1m.csv"
LOOKBACK_DAYS = 360    # same span for both assets (SPY has 2 years; set SPY_FULL=True as a secondary check only)
SPY_FULL = False

# --- economics (round trip, % of notional) ---
COST_PCT = {"btc": 0.085, "spy": 0.01}[ASSET]

# --- pre-registered experiment ---
ATR_KS   = [4.5, 7.5, 15.0]
HOLD     = 120
SEEDS    = [0, 1]
FOLDS    = 5

# --- model (identical to the BTC notebook) ---
ENC_LEN, MAX_EPOCHS, BATCH_SIZE, LR = 60, 30, 1024, 5e-4
D_MODEL, NHEAD, N_LAYERS, DROPOUT, PATIENCE = 64, 4, 2, 0.1, 7
FLOW_FAMILIES = {}     # price only

import torch
USE_GPU   = torch.cuda.is_available()
ACCEL     = "gpu" if USE_GPU else "cpu"
PRECISION = "16-mixed" if USE_GPU else "32-true"
print(f"ASSET={ASSET} | cost {COST_PCT}% | device: {ACCEL}" + (f" ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

In [ ]:
import time, gc, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
__import__("logging").getLogger("lightning.pytorch").setLevel(__import__("logging").ERROR)
if USE_GPU: torch.set_float32_matmul_precision("high")
try:
    from numba import njit; HAVE_NUMBA = True
except ImportError:
    HAVE_NUMBA = False
    def njit(*a, **k):
        def deco(f): return f
        return deco
    print("numba missing -> labels will be slow. pip install numba")
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()} | numba {HAVE_NUMBA}")

## 1. Load data (one asset per run)

In [ ]:
from google.cloud import bigquery

def fetch_once(lookback_days=360):
    client = bigquery.Client(project=BQ_PROJECT)
    q = f"""
    WITH bf AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, open, high, low, close, candle_total_vol, candle_delta,
               candle_total_trades, mark_price, funding_rate,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef') WHERE rn = 1),
    agg AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, vd_b2, vd_b3, vd_b4, vd_b5, vd_b6, vd_b7, vd_b8, vd_b9,
               vd_b10, vd_b11, net_liq, liq_total, oi_close,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef:bybitf') WHERE rn = 1)
    SELECT bf.ts AS timestamp, bf.open, bf.high, bf.low, bf.close,
           bf.candle_total_vol AS volume, bf.candle_delta, bf.candle_total_trades,
           bf.funding_rate, bf.mark_price,
           agg.vd_b2, agg.vd_b3, agg.vd_b4, agg.vd_b5, agg.vd_b6, agg.vd_b7,
           agg.vd_b8, agg.vd_b9, agg.vd_b10, agg.vd_b11,
           agg.net_liq, agg.liq_total, agg.oi_close
    FROM bf JOIN agg USING (ts)
    WHERE bf.close > 0
      AND bf.ts >= TIMESTAMP_SUB((SELECT MAX(ts) FROM `{BQ_TABLE}`),
                                 INTERVAL {lookback_days} DAY)
    ORDER BY bf.ts
    """
    print("Querying BigQuery (once)...")
    df = client.query(q).to_dataframe()
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["vd_retail"] = df[["vd_b2","vd_b3"]].sum(axis=1)
    df["vd_mid"]    = df[[f"vd_b{i}" for i in range(4,10)]].sum(axis=1)
    df["vd_whale"]  = df[["vd_b10","vd_b11"]].sum(axis=1)
    df = df.drop(columns=[f"vd_b{i}" for i in range(2,12)])
    print(f"Loaded {len(df):,} bars: {df.timestamp.min()} -> {df.timestamp.max()}")
    return df


def load_spy(path=SPY_CSV, lookback_days=LOOKBACK_DAYS, full=SPY_FULL):
    d = pd.read_csv(path)
    d["ts"] = pd.to_datetime(d["ts"], utc=True)
    et = d["ts"].dt.tz_convert("America/New_York")
    t = et.dt.time
    d = d[(t >= pd.Timestamp("09:30").time()) & (t < pd.Timestamp("16:00").time())].copy()
    d["timestamp"] = et[d.index].dt.tz_localize(None)       # exchange time, so hour features mean the same every day
    d = d.sort_values("timestamp").reset_index(drop=True)
    if not full:
        d = d[d["timestamp"] >= d["timestamp"].max() - pd.Timedelta(days=lookback_days)].reset_index(drop=True)
    print(f"Loaded {len(d):,} SPY regular-session bars: {d.timestamp.min()} -> {d.timestamp.max()}")
    return d[["timestamp","open","high","low","close","volume"]]


if ASSET == "btc":
    RAW = fetch_once(LOOKBACK_DAYS)[["timestamp","open","high","low","close","volume"]]
else:
    RAW = load_spy()
RAW = RAW.reset_index(drop=True)
print(RAW.tail(2))

## 2. Features (unchanged from the BTC notebook)

In [ ]:
def price_features(df):
    df = df.copy()
    for p in [1,5,15,30,60]: df[f"returns_{p}m"] = df["close"].pct_change(p)
    rng = df["high"] - df["low"] + 1e-10
    df["high_low_ratio"]   = df["high"]/df["low"]
    df["high_close_ratio"] = df["high"]/df["close"]
    df["low_close_ratio"]  = df["low"]/df["close"]
    df["upper_shadow"] = (df["high"]-np.maximum(df["open"],df["close"]))/rng
    df["lower_shadow"] = (np.minimum(df["open"],df["close"])-df["low"])/rng
    for p in [5,10,20,30]:
        sma = df["close"].rolling(p).mean()
        df[f"sma_{p}_slope"] = sma.pct_change()
        df[f"close_to_sma_{p}"] = (df["close"]-sma)/sma
    for p in [60,120]:
        sma = df["close"].rolling(p).mean()
        df[f"close_to_sma_{p}"] = (df["close"]-sma)/sma
    df["sma_120_slope"] = df["close"].rolling(120).mean().pct_change()
    e12, e26 = df["close"].ewm(span=12,adjust=False).mean(), df["close"].ewm(span=26,adjust=False).mean()
    macd = e12-e26
    df["macd_norm"] = macd/df["close"]
    df["macd_hist_norm"] = (macd-macd.ewm(span=9,adjust=False).mean())/df["close"]
    for p in [5,10,20,60]: df[f"volatility_{p}"] = df["returns_1m"].rolling(p).std()
    hl = df["high"]-df["low"]
    hc = (df["high"]-df["close"].shift()).abs()
    lc = (df["low"]-df["close"].shift()).abs()
    tr = np.maximum(hl, np.maximum(hc,lc))
    df["atr_60_abs"]  = tr.rolling(60).mean()
    df["atr_14_norm"] = tr.rolling(14).mean()/df["close"]
    df["atr_60_norm"] = df["atr_60_abs"]/df["close"]
    for p in [20,60]:
        sma, std = df["close"].rolling(p).mean(), df["close"].rolling(p).std()
        df[f"bb_position_{p}"] = (df["close"]-sma)/(2*std+1e-10)
        df[f"bb_width_{p}"] = std/sma
    def rsi(s,p):
        d = s.diff(); up = d.where(d>0,0).rolling(p).mean(); dn = (-d.where(d<0,0)).rolling(p).mean()
        return 100-100/(1+up/(dn+1e-10))
    for p in [14,20,60]: df[f"rsi_{p}"] = rsi(df["close"],p)
    for p in [14,60]:
        ll, hh = df["low"].rolling(p).min(), df["high"].rolling(p).max()
        df[f"stoch_k_{p}"] = 100*(df["close"]-ll)/(hh-ll+1e-10)
    df["stoch_d_14"] = df["stoch_k_14"].rolling(3).mean()
    df["volume_change"] = df["volume"].pct_change(1)
    for p in [5,10,20,60]:
        df[f"volume_ratio_{p}"] = df["volume"]/(df["volume"].rolling(p).mean()+1e-10)
    def mfi(d,p):
        tp = (d["high"]+d["low"]+d["close"])/3; mf = tp*d["volume"]
        pos = mf.where(tp>tp.shift(),0).rolling(p).sum()
        neg = mf.where(tp<tp.shift(),0).rolling(p).sum()
        return 100-100/(1+pos/(neg+1e-10))
    df["mfi_14"], df["mfi_60"] = mfi(df,14), mfi(df,60)
    for p in [20,60]:
        vw = (df["volume"]*df["close"]).rolling(p).sum()/(df["volume"].rolling(p).sum()+1e-10)
        df[f"close_to_vwap_{p}"] = (df["close"]-vw)/vw
    obv = (np.sign(df["close"].diff()).fillna(0)*df["volume"]).cumsum()
    df["obv_slope_20"] = obv.diff(20)/(df["volume"].rolling(20).sum()+1e-10)
    h,m,d_ = df.timestamp.dt.hour, df.timestamp.dt.minute, df.timestamp.dt.dayofweek
    df["hour_sin"],df["hour_cos"] = np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)
    df["minute_sin"],df["minute_cos"] = np.sin(2*np.pi*m/60), np.cos(2*np.pi*m/60)
    df["day_sin"],df["day_cos"] = np.sin(2*np.pi*d_/7), np.cos(2*np.pi*d_/7)
    return df


def flow_features(df):
    """Builds ALL flow families; selection happens later by column name."""
    df = df.copy(); vol = df["volume"]+1e-10
    for c in ["vd_retail","vd_mid","vd_whale"]: df[f"{c}_ratio"] = df[c]/vol
    df["vd_whale_vs_retail"] = df["vd_whale_ratio"]-df["vd_retail_ratio"]
    df["candle_delta_ratio"] = df["candle_delta"]/vol
    ats = df["volume"]/(df["candle_total_trades"]+1e-10)
    df["avg_trade_size_ratio"] = ats/(ats.rolling(60).mean()+1e-10)
    df["liq_intensity"] = df["liq_total"]/(df["liq_total"].rolling(240).mean()+1e-10)
    df["liq_imbalance"] = df["net_liq"]/(df["liq_total"]+1e-10)
    df["liq_vs_volume"] = df["liq_total"]/vol
    df["oi_change_1m"]  = df["oi_close"].pct_change(1)
    df["oi_change_15m"] = df["oi_close"].pct_change(15)
    oi_sma = df["oi_close"].rolling(240).mean()
    df["oi_vs_sma"] = (df["oi_close"]-oi_sma)/(oi_sma+1e-10)
    fr = df["funding_rate"]
    df["funding_z"] = (fr-fr.rolling(480).mean())/(fr.rolling(480).std()+1e-10)
    df["mark_dislocation_bps"] = (df["close"]-df["mark_price"])/df["close"]*1e4
    return df.drop(columns=["vd_retail","vd_mid","vd_whale","candle_delta",
                            "candle_total_trades","net_liq","liq_total",
                            "oi_close","mark_price"])


def barrier_labels(df, up_usd, dn_usd, hold, atr_mult=None):
    """First touch of +up_usd vs -dn_usd within `hold` bars. 1=up first, 0=down first."""
    close, high, low = df["close"].to_numpy(), df["high"].to_numpy(), df["low"].to_numpy()
    n = len(df)
    if atr_mult is not None:
        a = df["atr_60_abs"].to_numpy()
        ups, dns = a*atr_mult, a*atr_mult
    else:
        ups = np.full(n, float(up_usd)); dns = np.full(n, float(dn_usd))
    out = np.full(n, np.nan, dtype="float32")
    for i in range(n-1):
        u, d = ups[i], dns[i]
        if not (np.isfinite(u) and np.isfinite(d)) or u <= 0 or d <= 0: continue
        hi_t, lo_t = close[i]+u, close[i]-d
        end = min(i+hold, n-1)
        for j in range(i+1, end+1):
            hu, hd = high[j] >= hi_t, low[j] <= lo_t
            if hu and hd: break                 # ambiguous bar — discard
            if hu: out[i] = 1.0; break
            if hd: out[i] = 0.0; break
    return out

## 3. Model (unchanged from the BTC notebook)

In [ ]:
DEV = torch.device("cuda" if USE_GPU else "cpu")

class Bank:
    def __init__(self, X, y, enc):
        self.X = torch.from_numpy(X).to(DEV); self.y = torch.from_numpy(y).to(DEV)
        self.enc = enc; self.n = len(X)-enc+1
        self.off = torch.arange(enc, device=DEV)
    def gather(self, idx):
        return self.X[idx.unsqueeze(1)+self.off.unsqueeze(0)], self.y[idx+self.enc-1]

class IdxDS(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i): return i

class Clf(pl.LightningModule):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2,
                 dropout=0.1, learning_rate=5e-4, focal_alpha=0.5, focal_gamma=1.5):
        super().__init__(); self.save_hyperparameters()
        self.bank_train = self.bank_val = None
        self.inp = nn.Linear(n_features, d_model)
        self.pos = nn.Parameter(torch.randn(1,256,d_model)*0.02)
        lay = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                dim_feedforward=d_model*4, dropout=dropout,
                batch_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(lay, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model,d_model//2),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model//2,1))
    def forward(self,x):
        h = self.inp(x); h = h + self.pos[:,:h.size(1),:]
        return self.head(self.tr(h)[:,-1,:]).squeeze(-1)
    def _loss(self,lg,y):
        bce = nn.functional.binary_cross_entropy_with_logits(lg,y,reduction="none")
        p = torch.sigmoid(lg); pt = p*y+(1-p)*(1-y)
        at = self.hparams.focal_alpha*y+(1-self.hparams.focal_alpha)*(1-y)
        return (at*(1-pt)**self.hparams.focal_gamma*bce).mean()
    def _step(self,idx,bank,stage):
        x,y = bank.gather(idx); lg = self(x); loss = self._loss(lg,y)
        self.log(f"{stage}_loss", loss, batch_size=len(idx))
        return loss
    def training_step(self,i,_):   return self._step(i,self.bank_train,"train")
    def validation_step(self,i,_): return self._step(i,self.bank_val,"val")
    def configure_optimizers(self):
        o = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)
        return [o],[torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=self.trainer.max_epochs)]
print("model classes ready.")

## 4. Labels — first touch of ±barrier, per-bar barrier size, session-aware

Barrier for bar *i* is `up[i]`/`dn[i]` (fractions). Timeouts exit at market with real P&L.
`last[i]` = last bar a trade opened at *i* may use: end of data for BTC, the session's final bar for SPY.
Ambiguous bars (both barriers inside one bar) count as the stop — conservative, as before.

In [ ]:
@njit(cache=True)
def _first_touch_v(close, high, low, up, dn, last, max_hold):
    n = len(close)
    lab = np.zeros(n, dtype=np.int8); pnl = np.full(n, np.nan)
    res = np.zeros(n, dtype=np.int8); bars = np.full(n, np.nan)
    for i in range(n-1):
        u = up[i]; d = dn[i]
        if not (u > 0 and d > 0): continue                     # NaN barrier (warm-up) -> leave pnl NaN
        end = min(i + max_hold, last[i])
        if end <= i: continue                                  # opened on the session's last bar
        hi_t = close[i]*(1.0+u); lo_t = close[i]*(1.0-d); done = False
        for j in range(i+1, end+1):
            hu = high[j] >= hi_t; hd = low[j] <= lo_t
            if hd:                                             # stop first (also covers ambiguous bars)
                lab[i] = 0; pnl[i] = -d; res[i] = 1; bars[i] = j-i; done = True; break
            if hu:
                lab[i] = 1; pnl[i] =  u; res[i] = 1; bars[i] = j-i; done = True; break
        if not done:
            r = close[end]/close[i] - 1.0
            lab[i] = 1 if r > 0 else 0; pnl[i] = r; res[i] = 0; bars[i] = end-i
    return lab, pnl, res, bars


def last_index(df):
    n = len(df)
    if ASSET != "spy": return np.full(n, n-1, dtype=np.int64)
    day = df["timestamp"].dt.date.values
    last = np.empty(n, dtype=np.int64); j = n-1
    for i in range(n-1, -1, -1):
        if i < n-1 and day[i] != day[i+1]: j = i
        last[i] = j
    return last


def barrier_labels_v(df, mode, a, b, hold):
    """mode='pct': a,b are % (fixed).  mode='atr': a,b are ATR(60) multiples (per bar)."""
    n = len(df)
    if mode == "pct":
        up = np.full(n, a/100.0); dn = np.full(n, b/100.0)
    else:
        atr = df["atr_60_norm"].to_numpy(np.float64)
        up, dn = atr*a, atr*b
    lab, pnl, res, bars = _first_touch_v(df["close"].to_numpy(np.float64), df["high"].to_numpy(np.float64),
                                         df["low"].to_numpy(np.float64), up, dn, last_index(df), int(hold))
    return lab.astype("float32"), pnl, res.astype(bool), bars
print("labels ready" + ("  (numba on)" if HAVE_NUMBA else ""))

## 5. Runner — alignment fixed (everything read from `dva`)

In [ ]:
def build_frame_v(mode, a, b, hold):
    df = price_features(RAW)
    lab, pnl, res, bars = barrier_labels_v(df, mode, a, b, hold)
    df["target"], df["pnl_pct"], df["resolved"], df["bars"] = lab, pnl, res, bars
    df = df.replace([np.inf,-np.inf], np.nan)
    meta = ("timestamp","target","pnl_pct","resolved","bars")
    fc = [c for c in df.columns if c not in meta]
    df = df.dropna(subset=fc).dropna(subset=["pnl_pct"]).reset_index(drop=True)
    excl = set(meta) | {"open","high","low","close","volume","atr_60_abs"}
    feats = [c for c in df.columns if c not in excl and pd.api.types.is_numeric_dtype(df[c])]
    corr = df[feats].corr().abs()
    ut = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
    feats = [c for c in feats if not any(ut[c] > 0.95)]
    return df, feats


def run_v(mode, a, b, hold=HOLD, folds=FOLDS, seed=0, verbose=True):
    pl.seed_everything(seed, verbose=False)
    df, feats = build_frame_v(mode, a, b, hold)
    unit = "%" if mode == "pct" else "xATR"
    name = f"{ASSET} {a}{unit}/{b}{unit} h{hold} s{seed}"
    out = []
    for k in range(folds):
        f0 = 0.5 + k*(0.5/folds)
        tr_end, va_end = int(len(df)*f0), int(len(df)*(f0+0.5/folds))
        dtr, dva = df.iloc[:tr_end], df.iloc[tr_end:va_end]
        if len(dva) < ENC_LEN*3: continue
        sc = StandardScaler().fit(dtr[feats].values)
        btr = Bank(sc.transform(dtr[feats].values).astype("float32"), dtr["target"].values.astype("float32"), ENC_LEN)
        bva = Bank(sc.transform(dva[feats].values).astype("float32"), dva["target"].values.astype("float32"), ENC_LEN)
        m = Clf(len(feats), D_MODEL, NHEAD, N_LAYERS, DROPOUT, LR); m.bank_train, m.bank_val = btr, bva
        coll = lambda x: torch.tensor(x, device=DEV)
        tdl = DataLoader(IdxDS(btr.n), batch_size=BATCH_SIZE, shuffle=True, collate_fn=coll, drop_last=True)
        vdl = DataLoader(IdxDS(bva.n), batch_size=BATCH_SIZE, shuffle=False, collate_fn=coll)
        ck = ModelCheckpoint(dirpath=f"ckpt_ctrl/{ASSET}_{mode}_{a}_{hold}_{seed}_{k}", monitor="val_loss", save_top_k=1, mode="min")
        pl.Trainer(max_epochs=MAX_EPOCHS, accelerator=ACCEL, devices=1, precision=PRECISION, gradient_clip_val=1.0,
                   enable_progress_bar=False, enable_model_summary=False, logger=False,
                   callbacks=[EarlyStopping("val_loss", patience=PATIENCE, mode="min"), ck]).fit(m, tdl, vdl)
        best = Clf.load_from_checkpoint(ck.best_model_path, n_features=len(feats)).eval().to(DEV)
        P = []
        with torch.no_grad():
            for idx in vdl:
                x,_ = bva.gather(idx); P.append(torch.sigmoid(best(x)).float().cpu())
        pr = torch.cat(P).numpy()
        o = ENC_LEN - 1                                   # Bank window w predicts dva row w+ENC_LEN-1
        lb  = dva["target"].values[o:o+len(pr)]
        pnl = dva["pnl_pct"].values[o:o+len(pr)]*100
        rez = dva["resolved"].values[o:o+len(pr)]
        bar = dva["bars"].values[o:o+len(pr)]
        trade = np.where(pr > 0.5, pnl, -pnl) - COST_PCT
        conf = np.abs(pr-0.5); base = lb.mean(); const = max(base, 1-base)
        acc = ((pr > 0.5) == lb).mean(); med_bars = float(np.nanmedian(bar))
        out.append(dict(fold=k, n=len(lb), n_eff=max(1, int(len(lb)/max(1.0, med_bars))), resolved=rez.mean(),
                        med_bars=med_bars, base=base, acc=acc, edge=(acc-const)*100, ev=trade.mean(),
                        ev70=trade[conf>=0.20].mean() if (conf>=0.20).sum() > 10 else np.nan,
                        barrier_pct=float(np.nanmedian(np.abs(dva["pnl_pct"].values[o:o+len(pr)][rez])))*100 if rez.any() else np.nan))
    if not out: return None
    A = lambda key: float(np.nanmean([x[key] for x in out]))
    r = dict(asset=ASSET, mode=mode, a=a, b=b, hold=hold, seed=seed, name=name, folds=len(out),
             resolved=A("resolved"), med_bars=A("med_bars"), n_eff=int(A("n_eff")), barrier_pct=A("barrier_pct"),
             acc=A("acc"), edge=A("edge"), edge_min=float(min(x["edge"] for x in out)),
             pos_folds=int(sum(x["edge"] > 0 for x in out)), ev=A("ev"), ev70=A("ev70"))
    if verbose:
        se = np.sqrt(0.25/max(1, r["n_eff"]))*100
        print(f"{name:<28} barrier~{r['barrier_pct']:.2f}% resolv={r['resolved']*100:3.0f}% medBars={r['med_bars']:5.0f} "
              f"n_eff={r['n_eff']:>5} edge={r['edge']:+5.2f}pp (±{se:.2f}, worst {r['edge_min']:+5.2f}, "
              f"{r['pos_folds']}/{r['folds']} folds+) EV={r['ev']:+.3f}%")
    return r
print("run_v() ready.")

## 6. Reproduction check (BTC only) — should land near edge ≈ −0.9 pp

In [ ]:
REPRO = run_v("pct", 0.3, 0.3, hold=120, seed=0) if ASSET == "btc" else None
if REPRO: print("Trading (IV) reference: edge -0.89pp, EV -0.080%")

## 7. Pre-registered control run — 3 barrier sizes × 2 seeds × 5 folds

In [ ]:
RES = []
for k_ in ATR_KS:
    for s in SEEDS:
        r = run_v("atr", k_, k_, hold=HOLD, seed=s)
        if r: RES.append(r)
pd.DataFrame(RES).to_csv(f"results_{ASSET}.csv", index=False)
print(f"\nsaved results_{ASSET}.csv")

## 8. Side-by-side (run after both assets are done)

In [ ]:
frames = [pd.read_csv(f) for f in ("results_btc.csv","results_spy.csv") if os.path.exists(f)]
if frames:
    R = pd.concat(frames)
    g = R.groupby(["a","asset"]).agg(barrier_pct=("barrier_pct","mean"), resolved=("resolved","mean"),
            n_eff=("n_eff","mean"), edge=("edge","mean"), edge_sd=("edge","std"), worst=("edge_min","min"),
            folds_pos=("pos_folds","min"), ev=("ev","mean")).round(3)
    print(g.to_string())
    print("\nPASS = positive edge in all folds AND positive EV, on both seeds.")
    for (k_, asset), x in R.groupby(["a","asset"]):
        ok = (x.pos_folds == x.folds).all() and (x.ev > 0).all()
        print(f"  {asset} {k_}xATR: {'PASS' if ok else 'fail'}")